In [15]:
# svd can be done using scipy or numpy
import scipy.linalg as la
import numpy as np

Generate a random 5x4 matrix

In [16]:
np.random.seed(0) # to make the model reproducible

A = np.random.rand(5,4)
A

array([[0.5488135 , 0.71518937, 0.60276338, 0.54488318],
       [0.4236548 , 0.64589411, 0.43758721, 0.891773  ],
       [0.96366276, 0.38344152, 0.79172504, 0.52889492],
       [0.56804456, 0.92559664, 0.07103606, 0.0871293 ],
       [0.0202184 , 0.83261985, 0.77815675, 0.87001215]])

### Full SVD (`full_matrices = True` - default)

**input:**

$A \in \mathbb{R}^{m\times n}$

**output:**

$U \in \mathbb{R}^{m\times m}, S \in \mathbb{R}^{m\times n}, V \in \mathbb{R}^{n\times n}$

or, more precisely

$\boldsymbol{\sigma} \in \mathbb{R}^{q} $

where $q = \min(m,n)$.


In [17]:
U, s, VT = np.linalg.svd(A)
#U, s, VT = la.svd(A)
print('U shape: ', U.shape)
print('s shape: ', s.shape)
print('VT shape: ', VT.shape)

U shape:  (5, 5)
s shape:  (4,)
VT shape:  (4, 4)


Building matrix X

In [18]:
S = np.zeros(A.shape)
for i in range(len(s)):
    S[i, i] = s[i]
S

array([[2.64618677, 0.        , 0.        , 0.        ],
       [0.        , 0.83351254, 0.        , 0.        ],
       [0.        , 0.        , 0.70753001, 0.        ],
       [0.        , 0.        , 0.        , 0.29842614],
       [0.        , 0.        , 0.        , 0.        ]])

In [19]:
S = la.diagsvd(s, A.shape[0], A.shape[1])
S

array([[2.64618677, 0.        , 0.        , 0.        ],
       [0.        , 0.83351254, 0.        , 0.        ],
       [0.        , 0.        , 0.70753001, 0.        ],
       [0.        , 0.        , 0.        , 0.29842614],
       [0.        , 0.        , 0.        , 0.        ]])

Reconstructing matrix A

In [20]:
A_svd = np.matmul(U, np.matmul(S,VT))
# equivalently: A_svd = U @ S @ VT
# to check whether it's actual decompostion check the norm of the decompostion
print(f"err: {(la.norm(A - A_svd) / la.norm(A))}")

err: 4.3157440501506894e-16


### Thin SVD (`full_matrices = False`)

**input:**

$A \in \mathbb{R}^{m\times n}$

**output:**

$U \in \mathbb{R}^{m\times q}, S \in \mathbb{R}^{q\times q}, V \in \mathbb{R}^{n\times q}$

or, more precisely

$\boldsymbol{\sigma} \in \mathbb{R}^{q} $

where $q = \min(m,n)$.



In [21]:
U, s, VT = la.svd(A, full_matrices=False)
print('U shape: ', U.shape)
print('s shape: ', s.shape)
print('VT shape: ', VT.shape)

U shape:  (5, 4)
s shape:  (4,)
VT shape:  (4, 4)


In [22]:
S = np.diag(s)
S

array([[2.64618677, 0.        , 0.        , 0.        ],
       [0.        , 0.83351254, 0.        , 0.        ],
       [0.        , 0.        , 0.70753001, 0.        ],
       [0.        , 0.        , 0.        , 0.29842614]])

In [23]:
A_svd = np.matmul(U, np.matmul(S,VT))
print(f"err: {la.norm(A - A_svd) / la.norm(A)}")

err: 4.3157440501506894e-16


### A note on vectorization
Vectorization refers to the practice of replacing explicit loops with high-level mathematical operations that act on entire arrays or matrices at once. This leads to much better performance because it replaces slow Python loops with fast, optimized C and Fortran operations.

Indeed, we could be inclined to reconstruct $A_k$ with a for loop and the explicit formula
$$A_k = \sigma_1 u_1 v_1^T + ... + \sigma_k u_k v_k^T.$$

Let's measure the time taken for this operation for a matrix $A$ that is a bit larger

In [24]:
import time

A = np.random.rand(1000, 1500)
U, s, VT = la.svd(A, full_matrices=False)
S = np.diag(s)


Time the reconstruction with a for loop

In [25]:
start_time = time.time()

A_reconstructed_loop = np.zeros_like(A)
for i in range(len(s)):
    A_reconstructed_loop += s[i] * np.outer(U[:, i], VT[i, :])

loop_time = time.time() - start_time

Time the vectorized reconstruction using matrix multiplication

In [26]:
start_time = time.time()

A_reconstructed_matmult = U @ S @ VT

matmult_time = time.time() - start_time

If $S \in \mathbb R^{q \times q}$ diagonal (with elements on the diagonal equal to $s \in \mathbb R^{q}$), then the each of $US$ is equal to the element wise product of that row of $U$ with $s$, namely, $(US)_{i,j} = U_{i, j} s_j$. Indeed, $(US)_{i,j} = \sum_{k} U_{i, k} S_{k, j} = U_{i, j} S_{j, j} = U_{i, j} s_j$, since $S_{k, j} = 0$ iff $k \neq j$.

In [27]:
start_time = time.time()

# here we are using broadcasting to avoid the creation of a diagonal matrix
# see: https://numpy.org/doc/stable/user/basics.broadcasting.html
A_reconstructed_vectorized = (U * s) @ VT

vectorized_time = time.time() - start_time

We compare the results

In [28]:
print(f"Time for reconstruction using for loop: {loop_time:.6f} seconds")
print(f"Time for vectorized reconstruction: {vectorized_time:.6f} seconds")
print(f"Matmult is {loop_time / matmult_time:.1f} times faster than the loop")
print(f"Vectorized is {matmult_time / vectorized_time:.1f} times faster than the matmult")

difference = np.abs(A_reconstructed_loop - A_reconstructed_vectorized).max()
print(f"Difference between the two reconstructions: {difference:.6e}")

Time for reconstruction using for loop: 6.629597 seconds
Time for vectorized reconstruction: 0.097594 seconds
Matmult is 41.0 times faster than the loop
Vectorized is 1.7 times faster than the matmult
Difference between the two reconstructions: 5.662137e-15
